# BTC Macro Rubber Band Analysis
### Cointegration & Mean-Reversion Between BTC and Macro Assets
**Assets:** BTCUSDT vs DXY · Gold · EURUSD · SPX  
**Theory:** Inversely correlated assets diverge like a rubber band — find the stretch limit and reversion pattern.  
**Run cells top to bottom. Constants cell first — change parameters only there.**

## Cell 1 — Installs & Imports

In [ ]:
# ── Cell 1: Installs & Imports ─────────────────────────────────────────────
import subprocess, sys
for pkg in ['yfinance', 'statsmodels', 'hmmlearn', 'scikit-learn', 'scipy', 'seaborn', 'pykalman']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr

# Statsmodels — cointegration, ADF, OLS
from statsmodels.tsa.stattools import coint, adfuller, acf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.tsa.ar_model import AutoReg

# ML
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

# Kalman
from pykalman import KalmanFilter

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
plt.style.use('dark_background')
np.random.seed(42)

print('✅ All imports complete.')

## Cell 2 — Master Config
**All constants, tickers, paths, and thresholds live here. Change nothing elsewhere.**

In [ ]:
# ── Cell 2: Master Config ──────────────────────────────────────────────────

CFG = {
    # ── Tickers (yfinance) ────────────────────────────────────────────────
    'TICKER_BTC'    : 'BTC-USD',       # Bitcoin vs USD
    'TICKER_DXY'    : 'DX-Y.NYB',      # US Dollar Index
    'TICKER_GOLD'   : 'GC=F',          # Gold Futures (front month)
    'TICKER_EURUSD' : 'EURUSD=X',      # EUR/USD spot
    'TICKER_SPX'    : '^GSPC',         # S&P 500

    # ── Data Parameters ───────────────────────────────────────────────────
    # yfinance interval options: '1d', '1wk', '1mo'
    # Using weekly — long enough history for cointegration, avoids daily noise
    'INTERVAL'      : '1wk',
    'START_DATE'    : '2017-01-01',    # BTC has meaningful data from 2017
    'END_DATE'      : '2025-01-01',    # adjust as needed

    # ── Spread / Rubber Band Parameters ───────────────────────────────────
    # Rolling window for dynamic correlation (in bars)
    'CORR_WINDOW'   : 52,              # 52 weeks = 1 year rolling
    # Z-score threshold to flag "extreme stretch"
    'ZSCORE_THRESH' : 2.0,             # ±2σ = rubber band "too stretched"
    # Rolling window for spread z-score (in bars)
    'SPREAD_WINDOW' : 104,             # 2 years — long enough to capture regime
    # Half-life cap (weeks) — if reversion takes longer than this, ignore signal
    'HALFLIFE_CAP'  : 52,

    # ── Johansen Cointegration Test ───────────────────────────────────────
    # Rolling window for Johansen re-test (bars)
    'JOHANSEN_WINDOW': 104,            # 2-year rolling cointegration test
    # Confidence level: 0=90%, 1=95%, 2=99%
    'JOHANSEN_CL'   : 1,

    # ── HMM Parameters ────────────────────────────────────────────────────
    'HMM_STATES'    : 3,               # 3 regimes: bear / neutral / bull
    'HMM_ITER'      : 200,
    'HMM_FEATURES'  : ['spread_z', 'roll_corr', 'btc_ret', 'macro_ret'],

    # ── Visualization ─────────────────────────────────────────────────────
    'BG'            : '#0d0d0d',
    'TEAL'          : '#00e5cc',
    'ORANGE'        : '#ff6b35',
    'RED'           : '#ff2d55',
    'BLUE'          : '#4da6ff',
    'YELLOW'        : '#ffd700',
    'GREY'          : '#3a3a3a',
    'WHITE'         : '#e8e8e8',

    # ── Output Paths ──────────────────────────────────────────────────────
    'SAVE_FIGS'     : True,
    'FIG_PREFIX'    : 'btc_macro_',
}

# Asset display names and expected correlation direction with BTC
ASSET_META = {
    'DXY'    : {'label': 'US Dollar Index (DXY)',   'direction': -1,  'color': CFG['RED']},
    'GOLD'   : {'label': 'Gold (XAU/USD)',           'direction': +1,  'color': CFG['YELLOW']},
    'EURUSD' : {'label': 'EUR/USD',                  'direction': +1,  'color': CFG['TEAL']},
    'SPX'    : {'label': 'S&P 500',                  'direction': +1,  'color': CFG['BLUE']},
}
# direction: +1 = expected to move WITH BTC, -1 = expected to move AGAINST BTC

print('✅ Config loaded.')
print(f'   Interval : {CFG["INTERVAL"]}')
print(f'   Period   : {CFG["START_DATE"]} → {CFG["END_DATE"]}')
print(f'   Z-thresh : ±{CFG["ZSCORE_THRESH"]}σ (rubber band extreme)')
print(f'   Assets   : BTC vs {list(ASSET_META.keys())}')

## Cell 3 — Data Loader
Pulls all assets from yfinance. Aligns to common dates (intersection). Computes log returns and normalised price series for all assets.

In [ ]:
# ── Cell 3: Data Loader ────────────────────────────────────────────────────
# Robust downloader: tries primary ticker, then fallback tickers.
# Handles yfinance MultiIndex columns (new API) and missing/partial data.

# Fallback tickers for each asset (tried in order if primary fails)
TICKER_FALLBACKS = {
    'BTC'    : ['BTC-USD', 'BTC-USDT'],
    'DXY'    : ['DX-Y.NYB', 'DX=F', 'UUP'],          # UUP = USD ETF, last resort
    'GOLD'   : ['GC=F', 'XAUUSD=X', 'IAU', 'GLD'],   # spot FX, then ETFs
    'EURUSD' : ['EURUSD=X', 'EUR=X'],
    'SPX'    : ['^GSPC', 'SPY'],
}

def download_with_fallback(name, fallbacks, start, end, interval):
    """Try each ticker in fallbacks until one returns data."""
    for ticker in fallbacks:
        try:
            df = yf.download(
                ticker,
                start=start,
                end=end,
                interval=interval,
                progress=False,
                auto_adjust=True,
            )
            if df is None or df.empty:
                print(f'    [{ticker}] empty — trying next fallback...')
                continue

            # ── Handle MultiIndex (yfinance >= 0.2.x returns Price/Ticker MultiIndex)
            if isinstance(df.columns, pd.MultiIndex):
                # Level 0 = Price type (Open/High/Low/Close/Volume)
                # Level 1 = Ticker symbol
                if 'Close' in df.columns.get_level_values(0):
                    df = df['Close']
                    # If still MultiIndex (multiple tickers), take first column
                    if isinstance(df, pd.DataFrame):
                        df = df.iloc[:, 0]
                else:
                    # Flatten and find close column
                    df.columns = ['_'.join(c).strip() for c in df.columns]
                    close_cols = [c for c in df.columns if 'Close' in c or 'close' in c]
                    if not close_cols:
                        continue
                    df = df[close_cols[0]]
            else:
                # Flat columns — look for Close
                if 'Close' in df.columns:
                    df = df['Close']
                elif 'Adj Close' in df.columns:
                    df = df['Adj Close']
                else:
                    print(f'    [{ticker}] no Close column: {list(df.columns)}')
                    continue

            series = df.dropna()
            if len(series) < 50:
                print(f'    [{ticker}] only {len(series)} rows — too few, trying next...')
                continue

            print(f'  [{name:<6}] ✅ {ticker:<14} {len(series):,} bars | '
                  f'{series.index[0].date()} → {series.index[-1].date()}')
            return series

        except Exception as e:
            print(f'    [{ticker}] exception: {e}')
            continue

    print(f'  [{name}] ❌ ALL fallbacks failed: {fallbacks}')
    return None

# ── Download all assets ────────────────────────────────────────────────────
raw = {}
print('Downloading data from yfinance...')
for name, fallbacks in TICKER_FALLBACKS.items():
    # Use primary ticker from CFG as first in fallback list (already set above)
    series = download_with_fallback(
        name, fallbacks,
        CFG['START_DATE'], CFG['END_DATE'], CFG['INTERVAL']
    )
    if series is not None:
        raw[name] = series

print(f'\nSuccessfully loaded: {list(raw.keys())}')
missing = [k for k in TICKER_FALLBACKS if k not in raw]
if missing:
    print(f'⚠️  Missing assets: {missing} — they will be skipped in analysis')

# ── Align all loaded assets to common date index ───────────────────────────
price_df = pd.DataFrame({k: v for k, v in raw.items()})
price_df.index = pd.to_datetime(price_df.index)
price_df = price_df.sort_index()

# Report coverage before dropping NaNs
print('\nCoverage before alignment:')
for col in price_df.columns:
    non_null = price_df[col].dropna()
    print(f'  {col:<8}: {len(non_null):,} bars | {non_null.index[0].date()} → {non_null.index[-1].date()}')

price_df = price_df.dropna()   # keep only rows where ALL loaded assets have data

print(f'\nAligned dataset: {len(price_df):,} bars | {price_df.index[0].date()} → {price_df.index[-1].date()}')
print(f'Columns: {list(price_df.columns)}')

# Update ACTIVE_ASSETS — only analyse what we actually loaded
ACTIVE_ASSETS = [a for a in ['DXY', 'GOLD', 'EURUSD', 'SPX'] if a in price_df.columns]
print(f'Active assets for analysis: {ACTIVE_ASSETS}')

# ── Log returns ────────────────────────────────────────────────────────────
ret_df = np.log(price_df / price_df.shift(1)).dropna()

# ── Normalised price (rebased to 100 at start) ────────────────────────────
norm_df = (price_df / price_df.iloc[0]) * 100

print('\nPrice summary (raw):')
print(price_df.describe().round(4))


## Cell 4 — Utility Functions
Core math functions used across all analysis cells: spread computation, z-score, ADF test, half-life of mean reversion, Kalman hedge ratio.

In [ ]:
# ── Cell 4: Utility Functions ──────────────────────────────────────────────

def adf_test(series, label=''):
    """
    Augmented Dickey-Fuller test for stationarity.
    H0: series has unit root (non-stationary).
    Reject H0 (p<0.05) → series IS stationary → spread is mean-reverting.
    """
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'label'     : label,
        'adf_stat'  : result[0],
        'p_value'   : result[1],
        'stationary': result[1] < 0.05,
        'crit_1pct' : result[4]['1%'],
        'crit_5pct' : result[4]['5%'],
    }

def compute_half_life(spread):
    """
    Ornstein-Uhlenbeck half-life of mean reversion.
    Fit AR(1): Δspread = φ·spread_{t-1} + ε
    half_life = -ln(2) / ln(1 + φ)
    Faster reversion = shorter half-life = tighter rubber band snap.
    """
    spread = spread.dropna()
    delta  = spread.diff().dropna()
    lag    = spread.shift(1).dropna()
    # Align
    delta, lag = delta.align(lag, join='inner')
    model  = OLS(delta, add_constant(lag)).fit()
    phi    = model.params.iloc[1]   # AR coefficient on lag
    if phi >= 0 or phi <= -2:
        return np.nan   # no mean reversion
    half_life = -np.log(2) / np.log(1 + phi)
    return half_life

def compute_spread_zscore(btc_log, macro_log, beta, window):
    """
    Cointegrating spread z-score.
    spread = btc_log - beta * macro_log  (in log price space)
    z = (spread - rolling_mean) / rolling_std
    """
    spread = btc_log - beta * macro_log
    roll_mean = spread.rolling(window).mean()
    roll_std  = spread.rolling(window).std()
    z = (spread - roll_mean) / (roll_std + 1e-10)
    return spread, z

def kalman_hedge_ratio(btc_log, macro_log):
    """
    Kalman Filter for dynamic (time-varying) hedge ratio β.
    State vector: [β, α] — slope and intercept of regression BTC ~ β·MACRO + α.
    Avoids stale OLS β. Tracks the drift in cointegration relationship over time.
    Returns: beta_series (same index as inputs)
    """
    obs = btc_log.values
    states = macro_log.values
    n = len(obs)

    # Observation matrix: [macro_price, 1] each timestep
    obs_mat = np.vstack([states, np.ones(n)]).T[:, np.newaxis, :]

    kf = KalmanFilter(
        n_dim_obs=1,
        n_dim_state=2,
        initial_state_mean     = [0, 0],
        initial_state_covariance = np.eye(2),
        transition_matrices    = np.eye(2),
        observation_matrices   = obs_mat,
        observation_covariance = 1.0,
        transition_covariance  = 1e-3 * np.eye(2),
    )
    state_means, _ = kf.filter(obs)
    beta_series  = pd.Series(state_means[:, 0], index=btc_log.index, name='kalman_beta')
    alpha_series = pd.Series(state_means[:, 1], index=btc_log.index, name='kalman_alpha')
    return beta_series, alpha_series

def engle_granger_coint(btc_log, macro_log, label=''):
    """
    Engle-Granger two-step cointegration test.
    H0: No cointegration. Reject H0 (p<0.05) → cointegrated → rubber band holds.
    """
    score, pval, crits = coint(btc_log, macro_log)
    return {
        'label'        : label,
        'coint_stat'   : score,
        'p_value'      : pval,
        'cointegrated' : pval < 0.05,
        'crit_5pct'    : crits[1],
    }

def rolling_correlation(ret_a, ret_b, window):
    """Spearman rolling correlation (rank-based, handles crypto non-linearity)."""
    corr = []
    for i in range(window, len(ret_a)):
        a = ret_a.iloc[i-window:i].values
        b = ret_b.iloc[i-window:i].values
        r, _ = spearmanr(a, b)
        corr.append(r)
    idx = ret_a.index[window:]
    return pd.Series(corr, index=idx, name='roll_corr')

print('✅ Utility functions defined.')
print('   Functions: adf_test | compute_half_life | compute_spread_zscore')
print('              kalman_hedge_ratio | engle_granger_coint | rolling_correlation')

## Cell 5 — BTC vs DXY Analysis
Dollar Index is the primary inverse-correlation candidate. DXY up = USD strong = risk-off = BTC bearish pressure.  
Expected β direction: **negative** (if DXY rises, BTC falls).

In [ ]:
# ── Cell 5: BTC vs DXY ────────────────────────────────────────────────────

ASSET = 'DXY'
if ASSET not in price_df.columns:
    print(f'⚠️  {ASSET} not available — skipping.')
else:
    print(f'=== BTC vs {ASSET_META[ASSET]["label"]} ===')

    btc_log = np.log(price_df['BTC'])
    dxy_log = np.log(price_df[ASSET])
    btc_ret = ret_df['BTC']   # defined locally — safe across cell restarts
    dxy_ret = ret_df[ASSET]

    # ── 1. Static Pearson + Spearman correlation ───────────────────────────────
    aligned = pd.concat([btc_ret, dxy_ret], axis=1).dropna()
    pr, pp = pearsonr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    sr, sp = spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    print(f'\nStatic correlation (full period):')
    print(f'  Pearson  r = {pr:.4f}  p = {pp:.4f}  → {"significant" if pp<0.05 else "NOT significant"}')
    print(f'  Spearman r = {sr:.4f}  p = {sp:.4f}  → {"significant" if sp<0.05 else "NOT significant"}')

    # ── 2. Engle-Granger cointegration test ───────────────────────────────────
    coint_res = engle_granger_coint(btc_log, dxy_log, label='BTC/DXY')
    print(f'\nEngle-Granger cointegration:')
    print(f'  stat = {coint_res["coint_stat"]:.4f}  p = {coint_res["p_value"]:.4f}')
    print(f'  Result: {"✅ COINTEGRATED" if coint_res["cointegrated"] else "❌ NOT cointegrated"}')

    # ── 3. Kalman dynamic hedge ratio ─────────────────────────────────────────
    beta_dxy, alpha_dxy = kalman_hedge_ratio(btc_log, dxy_log)
    print(f'\nKalman hedge ratio β:')
    print(f'  Mean β = {beta_dxy.mean():.4f} | Min = {beta_dxy.min():.4f} | Max = {beta_dxy.max():.4f}')
    print(f'  (Negative β confirms inverse relationship)')

    # ── 4. Spread z-score using mean Kalman β ─────────────────────────────────
    beta_val = beta_dxy.mean()
    spread_dxy, zscore_dxy = compute_spread_zscore(
        btc_log, dxy_log, beta_val, CFG['SPREAD_WINDOW'])

    # ── 5. ADF on spread ──────────────────────────────────────────────────────
    adf_res = adf_test(spread_dxy, 'BTC/DXY spread')
    print(f'\nADF on spread (is it stationary = mean-reverting?):')
    print(f'  stat={adf_res["adf_stat"]:.4f}  p={adf_res["p_value"]:.4f}')
    print(f'  Result: {"✅ Spread is STATIONARY (mean-reverts)" if adf_res["stationary"] else "❌ Spread is NOT stationary"}')

    # ── 6. Half-life of mean reversion ───────────────────────────────────────
    hl_dxy = compute_half_life(spread_dxy)
    print(f'\nHalf-life of mean reversion: {hl_dxy:.1f} {CFG["INTERVAL"]} bars')
    if CFG['INTERVAL'] == '1wk':
        print(f'  = ~{hl_dxy/4.33:.1f} months to revert halfway to mean')

    # ── 7. Rolling correlation ────────────────────────────────────────────────
    roll_corr_dxy = rolling_correlation(btc_ret, dxy_ret, CFG['CORR_WINDOW'])

    # ── 8. Extreme stretch events ─────────────────────────────────────────────
    extreme_dxy = zscore_dxy[zscore_dxy.abs() > CFG['ZSCORE_THRESH']].dropna()
    print(f'\nRubber band extremes (|z| > {CFG["ZSCORE_THRESH"]}):')
    print(f'  Total extreme bars : {len(extreme_dxy)}')
    print(f'  Stretched LONG BTC : {(extreme_dxy < -CFG["ZSCORE_THRESH"]).sum()}  (DXY too high vs BTC → BTC should rise)')
    print(f'  Stretched SHORT BTC: {(extreme_dxy >  CFG["ZSCORE_THRESH"]).sum()}  (DXY too low vs BTC → BTC should fall)')

    RESULTS = {}
    RESULTS['DXY'] = {
        'pearson_r'        : pr,
        'spearman_r'       : sr,
        'cointegrated'     : coint_res['cointegrated'],
        'coint_p'          : coint_res['p_value'],
        'spread'           : spread_dxy,
        'zscore'           : zscore_dxy,
        'roll_corr'        : roll_corr_dxy,
        'half_life'        : hl_dxy,
        'beta'             : beta_dxy,
        'adf_stat'         : adf_res['adf_stat'],
        'adf_p'            : adf_res['p_value'],
        'spread_stationary': adf_res['stationary'],
        'extreme_events'   : extreme_dxy,
    }
    print('\n✅ DXY analysis complete. Stored in RESULTS["DXY"]')


## Cell 6 — BTC vs Gold Analysis
Gold is often described as BTC's closest macro analog — both treated as inflation hedges and store of value.  
Expected β direction: **positive** (BTC and Gold both rise in risk-off/inflation regimes).

In [ ]:
# ── Cell 6: BTC vs Gold ───────────────────────────────────────────────────

ASSET = 'GOLD'
if ASSET not in price_df.columns:
    print(f'⚠️  {ASSET} not available in price_df — skipping. Check Cell 3 output.')
else:
    print(f'=== BTC vs {ASSET_META[ASSET]["label"]} ===')

    btc_log  = np.log(price_df['BTC'])
    gold_log = np.log(price_df[ASSET])
    # Define locally — do not rely on Cell 5 scope
    btc_ret  = ret_df['BTC']
    gold_ret = ret_df[ASSET]

    # ── 1. Static correlations ────────────────────────────────────────────────
    aligned = pd.concat([btc_ret, gold_ret], axis=1).dropna()
    pr, pp = pearsonr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    sr, sp = spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    print(f'\nStatic correlation:')
    print(f'  Pearson  r = {pr:.4f}  p = {pp:.4f}  → {"significant" if pp<0.05 else "NOT significant"}')
    print(f'  Spearman r = {sr:.4f}  p = {sp:.4f}  → {"significant" if sp<0.05 else "NOT significant"}')

    # ── 2. Cointegration ──────────────────────────────────────────────────────
    coint_res = engle_granger_coint(btc_log, gold_log, label='BTC/GOLD')
    print(f'\nEngle-Granger cointegration:')
    print(f'  stat = {coint_res["coint_stat"]:.4f}  p = {coint_res["p_value"]:.4f}')
    print(f'  Result: {"✅ COINTEGRATED" if coint_res["cointegrated"] else "❌ NOT cointegrated"}')

    # ── 3. Kalman dynamic β ───────────────────────────────────────────────────
    beta_gold, alpha_gold = kalman_hedge_ratio(btc_log, gold_log)
    print(f'\nKalman hedge ratio β:')
    print(f'  Mean β = {beta_gold.mean():.4f} | Range: [{beta_gold.min():.2f}, {beta_gold.max():.2f}]')

    # ── 4. Spread + z-score ────────────────────────────────────────────────────
    beta_val = beta_gold.mean()
    spread_gold, zscore_gold = compute_spread_zscore(
        btc_log, gold_log, beta_val, CFG['SPREAD_WINDOW'])

    # ── 5. ADF ────────────────────────────────────────────────────────────────
    adf_res = adf_test(spread_gold, 'BTC/GOLD spread')
    print(f'\nADF on spread: stat={adf_res["adf_stat"]:.4f}  p={adf_res["p_value"]:.4f}')
    print(f'  {"✅ Stationary (mean-reverts)" if adf_res["stationary"] else "❌ Not stationary"}')

    # ── 6. Half-life ──────────────────────────────────────────────────────────
    hl_gold = compute_half_life(spread_gold)
    print(f'\nHalf-life: {hl_gold:.1f} weeks (~{hl_gold/4.33:.1f} months)')

    # ── 7. Rolling correlation ────────────────────────────────────────────────
    roll_corr_gold = rolling_correlation(btc_ret, gold_ret, CFG['CORR_WINDOW'])

    # ── 8. BTC/Gold ratio ─────────────────────────────────────────────────────
    btc_gold_ratio = price_df['BTC'] / price_df[ASSET]
    print(f'\nBTC/Gold ratio (oz of Gold per 1 BTC):')
    print(f'  Min: {btc_gold_ratio.min():.2f} oz | Max: {btc_gold_ratio.max():.2f} oz | Current: {btc_gold_ratio.iloc[-1]:.2f} oz')

    # ── 9. Extreme events ─────────────────────────────────────────────────────
    extreme_gold = zscore_gold[zscore_gold.abs() > CFG['ZSCORE_THRESH']].dropna()
    print(f'\nExtremes: {len(extreme_gold)} events | '
          f'+z: {(extreme_gold > CFG["ZSCORE_THRESH"]).sum()} | '
          f'-z: {(extreme_gold < -CFG["ZSCORE_THRESH"]).sum()}')

    # Store
    RESULTS['GOLD'] = {
        'pearson_r'        : pr,
        'spearman_r'       : sr,
        'cointegrated'     : coint_res['cointegrated'],
        'coint_p'          : coint_res['p_value'],
        'spread'           : spread_gold,
        'zscore'           : zscore_gold,
        'roll_corr'        : roll_corr_gold,
        'half_life'        : hl_gold,
        'beta'             : beta_gold,
        'adf_p'            : adf_res['p_value'],
        'spread_stationary': adf_res['stationary'],
        'extreme_events'   : extreme_gold,
        'btc_gold_ratio'   : btc_gold_ratio,
    }
    print('\n✅ Gold analysis complete. Stored in RESULTS["GOLD"]')


## Cell 7 — BTC vs EURUSD Analysis
EURUSD is the inverse of DXY (57.6% weight). When EURUSD rises = USD weakens = risk-on = BTC positive.  
Expected β direction: **positive**. This cross-validates the DXY result.

In [ ]:
# ── Cell 7: BTC vs EURUSD ─────────────────────────────────────────────────

ASSET = 'EURUSD'
if ASSET not in price_df.columns:
    print(f'⚠️  {ASSET} not available — skipping.')
else:
    print(f'=== BTC vs {ASSET_META[ASSET]["label"]} ===')

    btc_log    = np.log(price_df['BTC'])
    eurusd_log = np.log(price_df[ASSET])
    btc_ret    = ret_df['BTC']
    eurusd_ret = ret_df[ASSET]

    # ── 1. Static correlations ────────────────────────────────────────────────
    aligned = pd.concat([btc_ret, eurusd_ret], axis=1).dropna()
    pr, pp = pearsonr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    sr, sp = spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    print(f'\nStatic correlation:')
    print(f'  Pearson  r = {pr:.4f}  p = {pp:.4f}')
    print(f'  Spearman r = {sr:.4f}  p = {sp:.4f}')

    # ── 2. Cointegration ──────────────────────────────────────────────────────
    coint_res = engle_granger_coint(btc_log, eurusd_log, label='BTC/EURUSD')
    print(f'\nEngle-Granger cointegration:')
    print(f'  stat = {coint_res["coint_stat"]:.4f}  p = {coint_res["p_value"]:.4f}')
    print(f'  Result: {"✅ COINTEGRATED" if coint_res["cointegrated"] else "❌ NOT cointegrated"}')

    # ── 3. Kalman β ───────────────────────────────────────────────────────────
    beta_eur, alpha_eur = kalman_hedge_ratio(btc_log, eurusd_log)
    print(f'\nKalman β: mean={beta_eur.mean():.4f} | range=[{beta_eur.min():.2f}, {beta_eur.max():.2f}]')

    # ── 4. Spread + z-score ────────────────────────────────────────────────────
    beta_val = beta_eur.mean()
    spread_eur, zscore_eur = compute_spread_zscore(
        btc_log, eurusd_log, beta_val, CFG['SPREAD_WINDOW'])

    # ── 5. ADF ────────────────────────────────────────────────────────────────
    adf_res = adf_test(spread_eur, 'BTC/EURUSD spread')
    print(f'\nADF on spread: stat={adf_res["adf_stat"]:.4f}  p={adf_res["p_value"]:.4f}')
    print(f'  {"✅ Stationary" if adf_res["stationary"] else "❌ Not stationary"}')

    # ── 6. Half-life ──────────────────────────────────────────────────────────
    hl_eur = compute_half_life(spread_eur)
    print(f'\nHalf-life: {hl_eur:.1f} weeks (~{hl_eur/4.33:.1f} months)')

    # ── 7. Rolling correlation ────────────────────────────────────────────────
    roll_corr_eur = rolling_correlation(btc_ret, eurusd_ret, CFG['CORR_WINDOW'])

    # ── 8. DXY cross-check ────────────────────────────────────────────────────
    if 'DXY' in RESULTS:
        print(f'\nDXY cross-check:')
        print(f'  BTC/DXY Spearman    = {RESULTS["DXY"]["spearman_r"]:.4f}')
        print(f'  BTC/EURUSD Spearman = {sr:.4f}')
        sign_ok = RESULTS['DXY']['spearman_r'] * sr < 0
        print(f'  Sign consistency: {"✅ Opposite signs (correct)" if sign_ok else "⚠️ Same sign (unexpected)"}')

    # ── 9. Extreme events ─────────────────────────────────────────────────────
    extreme_eur = zscore_eur[zscore_eur.abs() > CFG['ZSCORE_THRESH']].dropna()
    print(f'\nExtremes: {len(extreme_eur)} events')

    RESULTS['EURUSD'] = {
        'pearson_r'        : pr,
        'spearman_r'       : sr,
        'cointegrated'     : coint_res['cointegrated'],
        'coint_p'          : coint_res['p_value'],
        'spread'           : spread_eur,
        'zscore'           : zscore_eur,
        'roll_corr'        : roll_corr_eur,
        'half_life'        : hl_eur,
        'beta'             : beta_eur,
        'adf_p'            : adf_res['p_value'],
        'spread_stationary': adf_res['stationary'],
        'extreme_events'   : extreme_eur,
    }
    print('\n✅ EURUSD analysis complete. Stored in RESULTS["EURUSD"]')


## Cell 8 — BTC vs SPX Analysis
Post-2020, BTC increasingly traded as a risk asset alongside equities. SPX correlation may exceed DXY correlation in recent years.  
Expected β direction: **positive** (risk-on = BTC up AND SPX up).

In [ ]:
# ── Cell 8: BTC vs SPX ────────────────────────────────────────────────────

ASSET = 'SPX'
if ASSET not in price_df.columns:
    print(f'⚠️  {ASSET} not available — skipping.')
else:
    print(f'=== BTC vs {ASSET_META[ASSET]["label"]} ===')

    btc_log = np.log(price_df['BTC'])
    spx_log = np.log(price_df[ASSET])
    btc_ret = ret_df['BTC']
    spx_ret = ret_df[ASSET]

    # ── 1. Static correlations ────────────────────────────────────────────────
    aligned = pd.concat([btc_ret, spx_ret], axis=1).dropna()
    pr, pp = pearsonr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    sr, sp = spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])
    print(f'\nStatic correlation:')
    print(f'  Pearson  r = {pr:.4f}  p = {pp:.4f}')
    print(f'  Spearman r = {sr:.4f}  p = {sp:.4f}')

    # ── 2. Cointegration ──────────────────────────────────────────────────────
    coint_res = engle_granger_coint(btc_log, spx_log, label='BTC/SPX')
    print(f'\nEngle-Granger cointegration:')
    print(f'  stat = {coint_res["coint_stat"]:.4f}  p = {coint_res["p_value"]:.4f}')
    print(f'  Result: {"✅ COINTEGRATED" if coint_res["cointegrated"] else "❌ NOT cointegrated"}')

    # ── 3. Kalman β ───────────────────────────────────────────────────────────
    beta_spx, alpha_spx = kalman_hedge_ratio(btc_log, spx_log)
    print(f'\nKalman β: mean={beta_spx.mean():.4f} | range=[{beta_spx.min():.2f}, {beta_spx.max():.2f}]')

    # ── 4. Spread + z-score ────────────────────────────────────────────────────
    beta_val = beta_spx.mean()
    spread_spx, zscore_spx = compute_spread_zscore(
        btc_log, spx_log, beta_val, CFG['SPREAD_WINDOW'])

    # ── 5. ADF ────────────────────────────────────────────────────────────────
    adf_res = adf_test(spread_spx, 'BTC/SPX spread')
    print(f'\nADF on spread: stat={adf_res["adf_stat"]:.4f}  p={adf_res["p_value"]:.4f}')
    print(f'  {"✅ Stationary" if adf_res["stationary"] else "❌ Not stationary"}')

    # ── 6. Half-life ──────────────────────────────────────────────────────────
    hl_spx = compute_half_life(spread_spx)
    print(f'\nHalf-life: {hl_spx:.1f} weeks (~{hl_spx/4.33:.1f} months)')

    # ── 7. Rolling correlation ────────────────────────────────────────────────
    roll_corr_spx = rolling_correlation(btc_ret, spx_ret, CFG['CORR_WINDOW'])

    # ── 8. Pre/Post 2020 regime split ─────────────────────────────────────────
    print('\nCorrelation regime split (pre vs post 2020):')
    pre2020  = ret_df[ret_df.index < '2020-01-01']
    post2020 = ret_df[ret_df.index >= '2020-01-01']
    for label, sub in [('Pre-2020', pre2020), ('Post-2020', post2020)]:
        if 'BTC' in sub.columns and 'SPX' in sub.columns and len(sub) > 20:
            r, p = spearmanr(sub['BTC'].dropna(), sub['SPX'].dropna())
            print(f'  {label}: Spearman r = {r:.4f}  p = {p:.4f}')

    # ── 9. Extreme events ─────────────────────────────────────────────────────
    extreme_spx = zscore_spx[zscore_spx.abs() > CFG['ZSCORE_THRESH']].dropna()
    print(f'\nExtremes: {len(extreme_spx)} events')

    RESULTS['SPX'] = {
        'pearson_r'        : pr,
        'spearman_r'       : sr,
        'cointegrated'     : coint_res['cointegrated'],
        'coint_p'          : coint_res['p_value'],
        'spread'           : spread_spx,
        'zscore'           : zscore_spx,
        'roll_corr'        : roll_corr_spx,
        'half_life'        : hl_spx,
        'beta'             : beta_spx,
        'adf_p'            : adf_res['p_value'],
        'spread_stationary': adf_res['stationary'],
        'extreme_events'   : extreme_spx,
    }
    print('\n✅ SPX analysis complete. Stored in RESULTS["SPX"]')


## Cell 9 — HMM Regime Detection
Hidden Markov Model fitted on each spread's z-score + rolling correlation.  
Identifies latent regimes: **Tight coupling / Stretched / Decoupled**.  
Run per asset — gives you a regime label at each timestep to understand WHEN the rubber band is active.

In [ ]:
# ── Cell 9: HMM Regime Detection ──────────────────────────────────────────

print('=== HMM Regime Detection ===')
print(f'States: {CFG["HMM_STATES"]} (0=bear/decoupled, 1=neutral, 2=bull/coupled)')

HMM_RESULTS = {}

for asset in ACTIVE_ASSETS:
    print(f'\n[{asset}]')
    try:
        zscore  = RESULTS[asset]['zscore'].dropna()
        roll_c  = RESULTS[asset]['roll_corr'].reindex(zscore.index).dropna()
        btc_r   = ret_df['BTC'].reindex(zscore.index).dropna()
        macro_r = ret_df[asset].reindex(zscore.index).dropna()

        # Align all series to common index
        common_idx = zscore.index.intersection(roll_c.index).intersection(btc_r.index).intersection(macro_r.index)
        features = pd.DataFrame({
            'zscore'    : zscore.reindex(common_idx),
            'roll_corr' : roll_c.reindex(common_idx),
            'btc_ret'   : btc_r.reindex(common_idx),
            'macro_ret' : macro_r.reindex(common_idx),
        }).dropna()

        if len(features) < CFG['HMM_STATES'] * 10:
            print(f'  Insufficient data ({len(features)} rows). Skipping.')
            continue

        # Standardise features
        scaler = StandardScaler()
        X = scaler.fit_transform(features.values)

        # Fit HMM
        model = GaussianHMM(
            n_components=CFG['HMM_STATES'],
            covariance_type='full',
            n_iter=CFG['HMM_ITER'],
            random_state=42,
        )
        model.fit(X)
        hidden_states = model.predict(X)

        states_series = pd.Series(hidden_states, index=features.index, name='regime')

        # Characterise each state by mean z-score and correlation
        state_stats = features.copy()
        state_stats['regime'] = hidden_states
        summary = state_stats.groupby('regime')[['zscore', 'roll_corr', 'btc_ret']].mean()
        summary['count'] = state_stats.groupby('regime').size()
        summary['pct']   = summary['count'] / len(state_stats) * 100

        print(f'  Regime statistics:')
        print(summary.round(4).to_string())
        print(f'  Log-likelihood: {model.score(X):.2f}')

        HMM_RESULTS[asset] = {
            'model'         : model,
            'states'        : states_series,
            'features'      : features,
            'state_stats'   : summary,
            'scaler'        : scaler,
        }
    except Exception as e:
        print(f'  HMM failed for {asset}: {e}')

print('\n✅ HMM complete. Stored in HMM_RESULTS')

## Cell 10 — Stretch Range Analysis
Measures the **empirical rubber band limits**: how far (in z-score units AND % price divergence) each pair stretches before mean-reverting.  
Computes: max historical stretch, percentile distribution, time-at-extreme, and forward return after extreme touch.

In [ ]:
# ── Cell 10: Stretch Range & Forward Return Analysis ──────────────────────

print('=== Rubber Band Stretch Range Analysis ===')

STRETCH = {}

for asset in ACTIVE_ASSETS:
    print(f'\n[{asset}]')
    z = RESULTS[asset]['zscore'].dropna()
    btc_close = price_df['BTC'].reindex(z.index)

    # ── Percentile distribution of z-score ─────────────────────────────
    pcts = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    pct_vals = np.percentile(z.dropna(), pcts)
    print(f'  Z-score percentiles:')
    print(f'  {dict(zip(pcts, pct_vals.round(3)))}')
    print(f'  Absolute max stretch: {z.abs().max():.2f}σ')
    print(f'  % time |z| > 2σ: {(z.abs() > 2).mean()*100:.1f}%')
    print(f'  % time |z| > 3σ: {(z.abs() > 3).mean()*100:.1f}%')

    # ── Forward return after extreme z-score ────────────────────────────
    fwd_records = []
    for thresh in [2.0, 2.5, 3.0]:
        for direction, label in [(z > thresh, f'+{thresh}σ'), (z < -thresh, f'-{thresh}σ')]:
            extreme_idx = z[direction].index
            for ts in extreme_idx:
                loc = z.index.get_loc(ts)
                for fwd in [4, 8, 13, 26]:  # 1mo, 2mo, 3mo, 6mo in weeks
                    if loc + fwd < len(btc_close):
                        ret = (btc_close.iloc[loc+fwd] / btc_close.iloc[loc] - 1) * 100
                        expected_dir = -1 if (z.iloc[loc] > 0 and asset == 'DXY') or \
                                             (z.iloc[loc] > 0 and asset != 'DXY') else 1
                        fwd_records.append({
                            'threshold': label,
                            'fwd_weeks': fwd,
                            'btc_ret'  : ret,
                            'z_val'    : z.iloc[loc],
                        })

    if fwd_records:
        fwd_df = pd.DataFrame(fwd_records)
        summary = fwd_df.groupby(['threshold', 'fwd_weeks'])['btc_ret'].agg(
            mean='mean', std='std', count='count',
            win_pct=lambda x: (x > 0).mean() * 100
        ).round(3)
        print(f'\n  Forward BTC return after z-score extreme:')
        print(summary.to_string())

    # ── Time at extreme (consecutive bars) ─────────────────────────────
    is_extreme = z.abs() > CFG['ZSCORE_THRESH']
    runs = []
    count = 0
    for val in is_extreme:
        if val: count += 1
        elif count > 0:
            runs.append(count)
            count = 0
    if runs:
        print(f'\n  Duration at extreme (consecutive weeks):')
        print(f'  Mean={np.mean(runs):.1f} | Max={np.max(runs)} | Median={np.median(runs):.1f}')

    STRETCH[asset] = {
        'pct_vals'  : dict(zip(pcts, pct_vals)),
        'max_z'     : z.abs().max(),
        'pct_2sd'   : (z.abs() > 2).mean() * 100,
        'pct_3sd'   : (z.abs() > 3).mean() * 100,
        'fwd_df'    : pd.DataFrame(fwd_records) if fwd_records else pd.DataFrame(),
        'run_lengths': runs,
    }

print('\n✅ Stretch analysis complete. Stored in STRETCH')

## Cell 11 — Visualization: Overview Dashboard
4-asset normalised price overlay + rolling correlations + spread z-scores over time.

In [ ]:
# ── Cell 11: Visualization — Overview Dashboard ───────────────────────────

BG, TEAL, ORG, RED, BLUE, YEL = (
    CFG['BG'], CFG['TEAL'], CFG['ORANGE'], CFG['RED'], CFG['BLUE'], CFG['YELLOW'])

fig = plt.figure(figsize=(22, 18), facecolor=BG)
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.3)
fig.suptitle('BTC Macro Rubber Band — Overview Dashboard', color='white', fontsize=15, y=0.98)

ax_colors = {'DXY': RED, 'GOLD': YEL, 'EURUSD': TEAL, 'SPX': BLUE, 'BTC': 'white'}

# ── Plot 1: Normalised price overlay (all assets rebased to 100) ───────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor(BG)
for col in ['BTC', 'DXY', 'GOLD', 'EURUSD', 'SPX']:
    ax1.plot(norm_df.index, norm_df[col],
             color=ax_colors[col], lw=1.5 if col == 'BTC' else 0.9,
             label=col, alpha=0.9)
ax1.set_yscale('log')
ax1.set_title('Normalised Price (rebased to 100 at start, log scale)', color='white', fontsize=11)
ax1.tick_params(colors='white')
ax1.legend(fontsize=9, labelcolor='white', facecolor=BG, ncol=5)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color(CFG['GREY'])

# ── Plot 2–5: Rolling correlation (BTC vs each asset) ─────────────────────
for i, asset in enumerate(['DXY', 'GOLD', 'EURUSD', 'SPX']):
    row = (i // 2) + 1
    col = i % 2
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor(BG)

    rc = RESULTS[asset]['roll_corr']
    color = ax_colors[asset]

    ax.plot(rc.index, rc.values, color=color, lw=1.2, alpha=0.85)
    ax.fill_between(rc.index, rc.values, 0,
                    where=(rc.values > 0), color=TEAL, alpha=0.15)
    ax.fill_between(rc.index, rc.values, 0,
                    where=(rc.values < 0), color=RED, alpha=0.15)
    ax.axhline(0, color=CFG['GREY'], lw=0.8, ls='--')
    ax.axhline(0.5, color=TEAL, lw=0.5, ls=':', alpha=0.5)
    ax.axhline(-0.5, color=RED, lw=0.5, ls=':', alpha=0.5)

    ax.set_ylim(-1, 1)
    ax.set_title(f'BTC / {asset} — Rolling {CFG["CORR_WINDOW"]}w Spearman Correlation',
                 color='white', fontsize=10)
    ax.set_ylabel('Correlation', color='white', fontsize=8)
    ax.tick_params(colors='white', labelsize=7)
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color(CFG['GREY'])

    # Annotate mean correlation
    ax.text(0.02, 0.05, f'mean r = {rc.mean():.3f}',
            transform=ax.transAxes, color=color, fontsize=8)

plt.savefig(f'{CFG["FIG_PREFIX"]}overview.png', dpi=150, bbox_inches='tight',
            facecolor=BG) if CFG['SAVE_FIGS'] else None
plt.show()
print('✅ Overview dashboard plotted.')

## Cell 12 — Visualization: Rubber Band Z-Score Charts
Spread z-score over time for each asset pair. Shaded extreme zones. Extreme events marked. This is the core rubber band visualization.

In [ ]:
# ── Cell 12: Rubber Band Z-Score Visualization ────────────────────────────

fig, axes = plt.subplots(4, 1, figsize=(22, 20), facecolor=BG)
fig.suptitle('Rubber Band Z-Score — How Far Have They Stretched?',
             color='white', fontsize=14, y=0.99)

thresh = CFG['ZSCORE_THRESH']

for ax, asset in zip(axes, ['DXY', 'GOLD', 'EURUSD', 'SPX']):
    ax.set_facecolor(BG)
    z = RESULTS[asset]['zscore'].dropna()
    color = ax_colors[asset]

    # Main z-score line
    ax.plot(z.index, z.values, color=color, lw=1.0, alpha=0.85, label='Spread Z-score')

    # Shaded extreme zones
    ax.fill_between(z.index, z.values, thresh,
                    where=(z.values > thresh), color=RED, alpha=0.35,
                    label=f'Overbought BTC vs {asset}')
    ax.fill_between(z.index, z.values, -thresh,
                    where=(z.values < -thresh), color=TEAL, alpha=0.35,
                    label=f'Oversold BTC vs {asset}')

    # Threshold lines
    ax.axhline(thresh,  color=RED,  lw=1, ls='--', alpha=0.7)
    ax.axhline(-thresh, color=TEAL, lw=1, ls='--', alpha=0.7)
    ax.axhline(thresh * 1.5, color=RED, lw=0.6, ls=':', alpha=0.5, label='±3σ')
    ax.axhline(-thresh * 1.5, color=TEAL, lw=0.6, ls=':', alpha=0.5)
    ax.axhline(0, color=CFG['GREY'], lw=0.8)

    # Mark extreme events with dots
    ext = RESULTS[asset]['extreme_events']
    if len(ext) > 0:
        ax.scatter(ext.index, ext.values,
                   color='white', s=15, zorder=5, alpha=0.7)

    # Stats box
    hl = RESULTS[asset]['half_life']
    coint = RESULTS[asset]['cointegrated']
    stat_text = (f"Half-life: {hl:.0f}w ({hl/4.33:.1f}mo)   "
                 f"Cointegrated: {'YES' if coint else 'NO'}   "
                 f"Extremes: {len(ext)}   "
                 f"Max stretch: {z.abs().max():.1f}σ")
    ax.text(0.01, 0.92, stat_text, transform=ax.transAxes,
            color='white', fontsize=8, alpha=0.9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=CFG['GREY'], alpha=0.5))

    ax.set_title(f'BTC vs {ASSET_META[asset]["label"]} — Spread Z-Score',
                 color='white', fontsize=10)
    ax.set_ylabel('Z-score (σ)', color='white', fontsize=8)
    ax.tick_params(colors='white', labelsize=7)
    ax.legend(fontsize=8, labelcolor='white', facecolor=BG, loc='upper right')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color(CFG['GREY'])

plt.savefig(f'{CFG["FIG_PREFIX"]}zscore.png', dpi=150, bbox_inches='tight',
            facecolor=BG) if CFG['SAVE_FIGS'] else None
plt.tight_layout()
plt.show()
print('✅ Z-score charts plotted.')

## Cell 13 — Visualization: HMM Regime Overlay + Kalman Beta Drift
Shows HMM regimes overlaid on BTC price. Shows how the hedge ratio β has drifted over time (critical — a constant β assumption is wrong).

In [ ]:
# ── Cell 13: HMM Regime Overlay + Kalman Beta Drift ───────────────────────

regime_colors = ['#ff2d55', '#888888', '#00e5cc']   # bear / neutral / bull regime

assets_with_hmm = [a for a in ['DXY', 'GOLD', 'EURUSD', 'SPX'] if a in HMM_RESULTS]
n = len(assets_with_hmm)

fig, axes = plt.subplots(n, 2, figsize=(22, 5*n), facecolor=BG)
if n == 1:
    axes = axes[np.newaxis, :]
fig.suptitle('HMM Regime Detection + Kalman Dynamic Hedge Ratio (β)',
             color='white', fontsize=13, y=1.01)

for i, asset in enumerate(assets_with_hmm):
    hmm   = HMM_RESULTS[asset]
    states = hmm['states']
    btc_p  = price_df['BTC'].reindex(states.index)
    beta   = RESULTS[asset]['beta'].reindex(states.index)

    # ── Left: BTC price with regime background ─────────────────────────
    ax_l = axes[i, 0]
    ax_l.set_facecolor(BG)

    # Shade background by regime
    for state_id in range(CFG['HMM_STATES']):
        mask = (states == state_id)
        ax_l.fill_between(states.index, btc_p.min(), btc_p.max(),
                          where=mask.values,
                          color=regime_colors[state_id], alpha=0.15)

    ax_l.plot(btc_p.index, btc_p.values, color='white', lw=1.2, alpha=0.9, label='BTC')
    ax_l.set_yscale('log')
    ax_l.set_title(f'BTC vs {asset} — HMM Regimes on BTC Price', color='white', fontsize=10)
    ax_l.tick_params(colors='white', labelsize=7)
    ax_l.spines[['top','right']].set_visible(False)
    ax_l.spines[['left','bottom']].set_color(CFG['GREY'])

    # Regime legend patches
    patches = [mpatches.Patch(color=regime_colors[j], alpha=0.5,
                               label=f'State {j} ({(states==j).mean()*100:.0f}%)')
               for j in range(CFG['HMM_STATES'])]
    ax_l.legend(handles=patches, fontsize=8, labelcolor='white',
                facecolor=BG, loc='upper left')

    # ── Right: Kalman β over time ──────────────────────────────────────
    ax_r = axes[i, 1]
    ax_r.set_facecolor(BG)
    color = ax_colors[asset]
    ax_r.plot(beta.index, beta.values, color=color, lw=1.2, alpha=0.9)
    ax_r.axhline(beta.mean(), color='white', lw=0.8, ls='--',
                 label=f'Mean β = {beta.mean():.3f}')
    ax_r.axhline(0, color=CFG['GREY'], lw=0.6)
    ax_r.fill_between(beta.index, beta.values, beta.mean(),
                      color=color, alpha=0.15)
    ax_r.set_title(f'BTC vs {asset} — Kalman Dynamic Hedge Ratio (β)',
                   color='white', fontsize=10)
    ax_r.set_ylabel('β', color='white', fontsize=9)
    ax_r.tick_params(colors='white', labelsize=7)
    ax_r.legend(fontsize=8, labelcolor='white', facecolor=BG)
    ax_r.spines[['top','right']].set_visible(False)
    ax_r.spines[['left','bottom']].set_color(CFG['GREY'])

plt.savefig(f'{CFG["FIG_PREFIX"]}hmm_kalman.png', dpi=150, bbox_inches='tight',
            facecolor=BG) if CFG['SAVE_FIGS'] else None
plt.tight_layout()
plt.show()
print('✅ HMM + Kalman charts plotted.')

## Cell 14 — Visualization: Stretch Distribution & Forward Return Heatmap
Shows the empirical distribution of how far each rubber band stretches. Heatmap of forward BTC returns after each extreme level.

In [ ]:
# ── Cell 14: Stretch Distribution & Forward Return Heatmap ────────────────

fig, axes = plt.subplots(2, 4, figsize=(22, 12), facecolor=BG)
fig.suptitle('Stretch Distribution & Forward Return by Threshold',
             color='white', fontsize=13)

for j, asset in enumerate(ACTIVE_ASSETS):
    color = ax_colors[asset]
    z = RESULTS[asset]['zscore'].dropna()

    # ── Top row: Z-score distribution histogram ───────────────────────
    ax_top = axes[0, j]
    ax_top.set_facecolor(BG)

    counts, bins, _ = ax_top.hist(
        z.values, bins=60, density=True,
        color=color, alpha=0.6, edgecolor='none')

    # Normal distribution overlay
    x_fit = np.linspace(z.min(), z.max(), 200)
    ax_top.plot(x_fit, stats.norm.pdf(x_fit, z.mean(), z.std()),
                color='white', lw=1.2, ls='--', alpha=0.7, label='Normal fit')

    ax_top.axvline(CFG['ZSCORE_THRESH'],  color=RED,  lw=1.5, ls='--')
    ax_top.axvline(-CFG['ZSCORE_THRESH'], color=TEAL, lw=1.5, ls='--')

    # Kurtosis annotation
    kurt = stats.kurtosis(z.dropna())
    skew = stats.skew(z.dropna())
    ax_top.text(0.05, 0.90, f'Kurt={kurt:.2f}\nSkew={skew:.2f}',
                transform=ax_top.transAxes, color='white', fontsize=8)

    ax_top.set_title(f'BTC/{asset} Z-Score Dist.', color='white', fontsize=10)
    ax_top.tick_params(colors='white', labelsize=7)
    ax_top.spines[['top','right']].set_visible(False)
    ax_top.legend(fontsize=7, labelcolor='white', facecolor=BG)

    # ── Bottom row: Forward return heatmap ────────────────────────────
    ax_bot = axes[1, j]
    ax_bot.set_facecolor(BG)

    fwd_data = STRETCH[asset].get('fwd_df', pd.DataFrame())
    if len(fwd_data) > 0:
        pivot = fwd_data.groupby(['threshold', 'fwd_weeks'])['btc_ret'].mean().unstack()
        sns.heatmap(
            pivot,
            ax=ax_bot,
            cmap='RdYlGn',
            center=0,
            annot=True,
            fmt='.1f',
            annot_kws={'size': 7, 'color': 'white'},
            cbar_kws={'label': 'Mean BTC ret %'},
            linewidths=0.5,
        )
        ax_bot.set_title(f'BTC/{asset} Fwd Return (%) after Extreme',
                         color='white', fontsize=9)
        ax_bot.set_xlabel('Fwd weeks', color='white', fontsize=8)
        ax_bot.set_ylabel('Threshold', color='white', fontsize=8)
        ax_bot.tick_params(colors='white', labelsize=7)
    else:
        ax_bot.text(0.3, 0.5, 'No extremes\ndetected',
                    color='white', transform=ax_bot.transAxes, fontsize=10)

plt.savefig(f'{CFG["FIG_PREFIX"]}stretch_dist.png', dpi=150, bbox_inches='tight',
            facecolor=BG) if CFG['SAVE_FIGS'] else None
plt.tight_layout()
plt.show()
print('✅ Distribution + heatmap plotted.')

## Cell 15 — Compiled Results Summary
Single structured table of all empirical findings across all 4 asset pairs. This is the calibration output — strength of rubber band, reliability, half-life, and which asset is the tightest pair with BTC.

In [ ]:
# ── Cell 15: Compiled Results Summary ────────────────────────────────────

sep = '='*75
print(sep)
print('  BTC MACRO RUBBER BAND — COMPILED ANALYSIS SUMMARY')
print(sep)

# ── 1. Correlation table ──────────────────────────────────────────────────
print('\n1. CORRELATION (full-period, weekly returns):')
print(f'{"Asset":<10} {"Direction":<12} {"Pearson r":<12} {"Spearman r":<13} {"Expected dir"}')
print('-'*65)
for asset in ACTIVE_ASSETS:
    exp_dir = ASSET_META[asset]['direction']
    actual_sign = np.sign(RESULTS[asset]['spearman_r'])
    match = '✅' if actual_sign == exp_dir else '⚠️  Unexpected'
    print(f"{asset:<10} {str(exp_dir):<12} "
          f"{RESULTS[asset]['pearson_r']:>+.4f}      "
          f"{RESULTS[asset]['spearman_r']:>+.4f}      {match}")

# ── 2. Cointegration table ────────────────────────────────────────────────
print('\n2. COINTEGRATION (Engle-Granger):')
print(f'{"Asset":<10} {"Cointegrated":<15} {"p-value":<12} {"Spread ADF p":<15} {"Stationary spread"}')
print('-'*70)
for asset in ACTIVE_ASSETS:
    r = RESULTS[asset]
    print(f"{asset:<10} {'YES ✅' if r['cointegrated'] else 'NO  ❌':<15} "
          f"{r['coint_p']:.4f}       "
          f"{r['adf_p']:.4f}          "
          f"{'YES ✅' if r['spread_stationary'] else 'NO  ❌'}")

# ── 3. Rubber band metrics ────────────────────────────────────────────────
print('\n3. RUBBER BAND METRICS:')
print(f'{"Asset":<10} {"Half-life(wk)":<15} {"Half-life(mo)":<15} {"Max stretch(σ)":<16} {"% time >2σ":<12} {"Extremes(n)"}')
print('-'*80)
for asset in ACTIVE_ASSETS:
    r  = RESULTS[asset]
    st = STRETCH[asset]
    hl = r['half_life']
    print(f"{asset:<10} {hl:.1f}{'':>9} "
          f"{hl/4.33:.1f}{'':>10} "
          f"{st['max_z']:.2f}{'':>11} "
          f"{st['pct_2sd']:.1f}%{'':>6} "
          f"{len(r['extreme_events'])}")

# ── 4. Verdict: Which rubber band is strongest? ───────────────────────────
print('\n4. RANKING — TIGHTEST RUBBER BAND (most reliable mean-reversion):')
scores = {}
for asset in ACTIVE_ASSETS:
    r = RESULTS[asset]
    # Score: cointegrated + stationary spread + |correlation| + short half-life
    score = 0
    score += 2 if r['cointegrated'] else 0
    score += 2 if r['spread_stationary'] else 0
    score += abs(r['spearman_r']) * 3
    hl = r['half_life']
    score += max(0, 3 - hl/52)  # bonus for half-life < 1 year
    scores[asset] = round(score, 3)

ranked = sorted(scores.items(), key=lambda x: -x[1])
for rank, (asset, score) in enumerate(ranked, 1):
    print(f'  #{rank} {asset:<8} composite score = {score:.3f}')

print('\n5. PRACTICAL INTERPRETATION:')
best = ranked[0][0]
hl_best = RESULTS[best]['half_life']
print(f'  Strongest rubber band: BTC vs {best}')
print(f'  Mean reversion half-life: ~{hl_best:.0f} weeks (~{hl_best/4.33:.1f} months)')
print(f'  When spread z > +{CFG["ZSCORE_THRESH"]}: BTC is stretched ABOVE {best} → mean-reversion bearish for BTC')
print(f'  When spread z < -{CFG["ZSCORE_THRESH"]}: BTC is stretched BELOW {best} → mean-reversion bullish for BTC')
print(f'\n  IMPORTANT CAVEATS:')
print(f'  • This is a HIGHER-TIMEFRAME signal (weekly) — NOT a scalp trigger')
print(f'  • Cointegration breaks during structural regime shifts (e.g. 2020 Fed QE)')
print(f'  • Use as a directional bias / confluence filter, not standalone entry')
print(f'  • Re-test cointegration rolling — relationship is not permanent')
print(f'\n{sep}')